In [ ]:
######################################## BASE CASE SIMULATION ########################################
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

############ Preparation
# Load the data
data = pd.read_csv("scooterdata.csv")

# Validate the data
if data.isnull().values.any():
    raise ValueError("Input data contains missing values.")
if not set(data['origin']).issubset(set("ABCDEFGH")) or not set(data['destination']).issubset(set("ABCDEFGH")):
    raise ValueError("Origin or Destination contains values outside the range A-H.")

required_columns = {'origin', 'destination', 'revenue', 'duration'}
demand_columns = {f'Demand{i}' for i in range(1, 101)}
missing_columns = required_columns.union(demand_columns) - set(data.columns)

# If all checks pass, print a confirmation message
print("Data validation passed. The dataset is ready for processing.")
print(data.head())

############ Variable setting
# Set random seed for reproducibility
np.random.seed(42)

# Initialize variables
stations = 8
total_scooters = 100
fixed_cost_per_scooter = 80
scooters_per_station = total_scooters // stations
remaining_scooters = total_scooters % stations
max_duration = int(data['duration'].max())
total_days = 500
time_slots = 100

# Arrays to store results
unmet_demand_per_timeslot = np.zeros((total_days, time_slots))
total_demand_per_timeslot = np.zeros((total_days, time_slots))
scooters_in_use_per_timeslot = np.zeros((total_days, time_slots))
total_scooters_per_timeslot = np.full((total_days, time_slots), total_scooters)
station_inflows = np.zeros((total_days, stations))
station_outflows = np.zeros((total_days, stations))

# Initialize station states
def initialize_station_states():
    station_states = np.zeros((stations, max_duration + 1), dtype=int)
    station_states[:4, 0] = 13 # Assign 13 scooters to the first 4 stations
    station_states[4:, 0] = 12 # Assign 12 scooters to the other 4 stations
    return station_states

############ Simulation logic
# Simulation for time and demand
def simulation_frame(data, station_states, total_scooters, day, seed=None):
    if seed is not None:
        np.random.seed(seed)

    revenue = 0
    total_demand = 0
    unmet_demand = 0
    scooters_in_use = 0
    inflows = np.zeros(stations)
    outflows = np.zeros(stations)
    total_demand_per_timeslot = np.zeros((total_days, time_slots))
    scooters_in_use_per_timeslot = np.zeros((total_days, time_slots))
    unmet_demand_per_timeslot = np.zeros((total_days, time_slots))

    for time in range(time_slots):  # Simulate 100 time slots
        for index, row in data.iterrows():
            origin = ord(row['origin']) - ord('A')
            destination = ord(row['destination']) - ord('A')
            demand_rate = row[f"Demand{time + 1}"]
            demand = np.random.poisson(demand_rate)
            trip_duration = int(row['duration'])
            trip_revenue = float(row['revenue'])
            total_demand += demand
            total_demand_per_timeslot[day, time] += demand

            for _ in range(demand):
                if station_states[origin, 0] > 0:
                    station_states[origin, 0] -= 1
                    station_states[destination, trip_duration] += 1
                    inflows[destination] += 1
                    outflows[origin] += 1
                    revenue += trip_revenue
                    scooters_in_use += 1
                    scooters_in_use_per_timeslot[day, time] += 1
                else:
                    unmet_demand += 1
                    unmet_demand_per_timeslot[day, time] += 1

        # Update station states
        station_states[:, :-1] = station_states[:, 1:]
        station_states[:, -1] = 0

    #Calculate daily utilization
    operating_cost = total_scooters * fixed_cost_per_scooter
    profit = revenue - operating_cost

    #return metrics
    return {
        "Revenue": revenue,
        "Profit": profit,
        "TotalDemand": total_demand,
        "TotalDemandTimeslot": total_demand_per_timeslot,
        "UnmetDemand": unmet_demand,
        "UnmetDemandTimeslot": unmet_demand_per_timeslot,
        "TotalscooterUsed": scooters_in_use,
        "TotalscooterUsedTimeslot": scooters_in_use_per_timeslot,
        "Inflows": inflows,
        "Outflows": outflows,
    }

#Result storage generation
daily_revenue = []
daily_profit = []
daily_revenue_per_scooter = []
daily_profit_per_scooter = []
unmet_demand_percentage_per_timeslot = np.zeros((total_days, time_slots))
scooter_utilization_percentage_per_timeslot = np.zeros((total_days, time_slots))
station_inflows = np.zeros((total_days, stations))
station_outflows = np.zeros((total_days, stations))

# Simulation for 500 days
for day in range(total_days):
    station_states = initialize_station_states()
    result = simulation_frame(data, station_states, total_scooters, day, seed = 42 + day)

    # Store daily metrics
    daily_revenue.append(result["Revenue"])
    daily_profit.append(result["Profit"])
    daily_revenue_per_scooter.append(result["Revenue"] / total_scooters)
    daily_profit_per_scooter.append(result["Profit"] / total_scooters)

    #Store scooter unavailability rate
    unmet_demand_percentage_per_timeslot[day, :] = np.where(
        result["TotalDemandTimeslot"][day, :] > 0,  # Use only the current day's data
        (result["UnmetDemandTimeslot"][day, :] / result["TotalDemandTimeslot"][day, :]) * 100,
        0
    )
    #Store scooter utilization rate
    scooter_utilization_percentage_per_timeslot[day, :] = (
       result["TotalscooterUsedTimeslot"][day, :] / total_scooters) * 100

    #Store in/outflow
    station_inflows[day, :] = result["Inflows"]
    station_outflows[day, :] = result["Outflows"]

    total_station_inflows_avg = station_inflows.mean(axis=0)
    total_station_outflows_avg = station_outflows.mean(axis=0)
    total_station_inflows_sum = station_inflows.sum(axis=0)
    total_station_outflows_sum = station_outflows.sum(axis=0)

############ Final metrics
# Daily metrics
daily_metrics = pd.DataFrame({
    "Day": range(1, total_days + 1),
    "DailyRevenue": daily_revenue,
    "DailyProfit": daily_profit,
    "RevenuePerScooter": daily_revenue_per_scooter,
    "ProfitPerScooter": daily_profit_per_scooter,
})
daily_metrics.to_csv("daily_metrics.csv", index=False)

# Station inflows and outflows
station_metrics = pd.DataFrame({
    "Station": [f"Station_{i+1}" for i in range(stations)],
    "TotalInflows": total_station_inflows_sum,
    "TotalOutflows": total_station_outflows_sum,
    "AverageInflows": total_station_inflows_avg,
    "AverageOutflows": total_station_outflows_avg,
})
station_metrics.to_csv("station_metrics.csv", index=False)

# Time slot averages
average_unmet_demand = unmet_demand_percentage_per_timeslot.mean(axis=0)
average_utilization =  scooter_utilization_percentage_per_timeslot.mean(axis=0)

axis1 = unmet_demand_percentage_per_timeslot.T.mean(axis=0)
axis2 = scooter_utilization_percentage_per_timeslot.T.mean(axis=0)
print(f'total shape:{scooter_utilization_percentage_per_timeslot.shape}')

total_day_metrics = pd.DataFrame({
    "Days": range(1, total_days + 1),
    "AverageUnmetDemandPercentage": axis1,
    "AverageUtilizationPercentage": axis2,
})
total_day_metrics.to_csv("total_day_metrics.csv", index=False)

time_slot_metrics = pd.DataFrame({
    "TimeSlot": range(1, time_slots + 1),
    "AverageUnmetDemandPercentage": average_unmet_demand,
    "AverageUtilizationPercentage": average_utilization,
})
time_slot_metrics.to_csv("time_slot_metrics.csv", index=False)

############ Visualisation
#Total revenue and profit
plt.figure(figsize = (10, 6))
plt.plot(range(1, total_days + 1), daily_revenue, label = "Daily Revenue (EUR)")
plt.plot(range(1, total_days + 1), daily_profit, label = "Daily profit (EUR)")
plt.xlabel("Day")
plt.ylabel("Amount")
plt.legend()
plt.grid()
plt.show()

#Per scooter revenue and profit
plt.figure(figsize = (10, 6))
plt.plot(range(1, total_days + 1), daily_revenue_per_scooter, label = "Revenue per scooter(EUR)")
plt.plot(range(1, total_days + 1), daily_profit_per_scooter, label = "Profit per scooter (EUR)")
plt.xlabel("Day")
plt.ylabel("Amount")
plt.legend()
plt.grid()
plt.show()

#Percentage of demand unmet
plt.figure(figsize = (10, 6))
plt.plot(average_unmet_demand, label = "Percentage of demand unmet")
plt.xlabel("Time slot")
plt.ylabel("Unmet demand %")
plt.legend()
plt.grid()
plt.show()

#Percentage of scooter utilization rate???????
plt.figure(figsize = (10, 6))
plt.plot(average_utilization, label = "Percentage of scooter utilization")
plt.xlabel("Time slot")
plt.ylabel("Scooter utilization %")
plt.legend()
plt.grid()
plt.show()

#In/outflow of scooter at each station
# Set bar width and positions
stations = [f"Station_{i+1}" for i in range(8)]
x = np.arange(len(stations))  # Indices for the stations
width = 0.35  # Width of the bars

# Total Inflows and Outflows
fig, ax = plt.subplots(figsize=(12, 6))
bars1 = ax.bar(x - width/2, total_station_inflows_sum, width, label="Total Inflows", color="blue", alpha=0.8)
bars2 = ax.bar(x + width/2, total_station_outflows_sum, width, label="Total Outflows", color="orange", alpha=0.8)

# Add values above the bars with commas
for bar in bars1:
    ax.text(bar.get_x() + bar.get_width() / 2, bar.get_height(), f'{int(bar.get_height()):,}', ha='center', va='bottom')
for bar in bars2:
    ax.text(bar.get_x() + bar.get_width() / 2, bar.get_height(), f'{int(bar.get_height()):,}', ha='center', va='bottom')

# Add labels and titles
ax.set_xlabel("Stations")
ax.set_ylabel("Total Values")
ax.set_xticks(x)
ax.set_xticklabels(stations)
ax.legend()

# Add grid and show the plot
plt.grid(axis="y", linestyle="--", alpha=0.7)
plt.tight_layout()
plt.show()

# Average Inflows and Outflows
fig, ax = plt.subplots(figsize=(12, 6))
bars1 = ax.bar(x - width/2, total_station_inflows_avg, width, label="Average Inflows", color="blue", alpha=0.8)
bars2 = ax.bar(x + width/2, total_station_outflows_avg, width, label="Average Outflows", color="orange", alpha=0.8)

# Add rounded values above the bars
for bar in bars1:
    ax.text(bar.get_x() + bar.get_width() / 2, bar.get_height(), f'{round(bar.get_height())}', ha='center', va='bottom')
for bar in bars2:
    ax.text(bar.get_x() + bar.get_width() / 2, bar.get_height(), f'{round(bar.get_height())}', ha='center', va='bottom')

# Add labels and titles
ax.set_xlabel("Stations")
ax.set_ylabel("Average Values")
ax.set_xticks(x)
ax.set_xticklabels(stations)
ax.legend()

# Add grid and show the plot
plt.grid(axis="y", linestyle="--", alpha=0.7)
plt.tight_layout()
plt.show()

